In [ ]:
%pylab inline

In [ ]:
import sys
import scipy
import numpy
import json
import bokeh
import tensorflow
import keras
import sklearn
import matplotlib
import pandas as pd
import seaborn as sns

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.0.0/spark-3.0.0-bin-hadoop3.2.tgz
!tar xf spark-3.0.0-bin-hadoop3.2.tgz
 
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.0.0-bin-hadoop3.2"

In [ ]:
pip install findspark

In [ ]:
import findspark
findspark.init()

In [ ]:
import pyspark
from pyspark.sql import SparkSession
sp = pyspark.SparkContext()
spark = SparkSession(sp)
print(sp)

In [ ]:
!pip install -U -q PyDrive # Pydrive helps in mouting dataset to collab

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials  #oauth2 support given
 
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default() # prompt to sign in
drive = GoogleDrive(gauth)

In [ ]:
file_list = drive.ListFile({'q': "'1fdHixmn1GUa_OXD3BQb5aJks8NQC2nHR' in parents and trashed=false"}).GetList() # folder_id posted here
for file1 in file_list:
  print('%s -> %s' % (file1['title'], file1['id'])) # List of files in the folder

In [ ]:
f_id=''
drive_file = drive.CreateFile({'id': f_id})
drive_file = drive.CreateFile({'id':'1Og7PpcAa1NfstsLdRL6efIYTDg8_lpvq'}) #file_id to mount
drive_file.GetContentFile('topd.csv') #getfile

f_id=''
drive_file = drive.CreateFile({'id': f_id})
drive_file = drive.CreateFile({'id':'1CAwm6CqvUZrVjouWpbdCu4uajFgk-Q-p'}) #file_id to mount
drive_file.GetContentFile('dns.csv') #getfile

In [ ]:
top_l=pd.read_csv("topd.csv",nrows=100000) #Read file
#print(top_l)
top_rdd = sp.parallelize(top_l['google.com'], 4)
#top_rdd.collect()

In [ ]:
dga_p = pd.read_csv('dns.csv',nrows=100000)
print(dga_p)
dga_l = dga_p['dns1.viphone.cz.'].values.tolist()
dga_rdd = sp.parallelize(dga_p['dns1.viphone.cz.'], 4)
#dga_rdd.collect()

In [ ]:
def filt(word):
    return ('[a-zA-Z0-9]*.' +word)

In [ ]:
domain_filt_words = top_rdd.map(filt)
print(domain_filt_words.collect())

In [ ]:
pairkeynames = (domain_filt_words
                  .map(lambda na: (na, 1))
                  .reduceByKey(lambda o,p: o+p)
                  .collect())
print (pairkeynames)

In [ ]:
d_w = domain_filt_words.map(lambda na: (na, 1))

In [ ]:
deptColumns = ["domain_name","count"]
df2 = d_w.toDF(deptColumns)
df2.printSchema()
df2.show(truncate=False)

p_df = df2.toPandas()
domain_df = p_df['domain_name']

domain_li = domain_df.values.tolist()
print(domain_li)

In [ ]:
import re
def reducer():
  to_rem = []
  for i in dga_l:
    for j in domain_li:
      emails = re.findall(j, i)
      for email in emails:
        to_rem.append(str(email))
  print(to_rem)
  return to_rem

In [ ]:
to_rem = (non_rep_doms.map(lambda w: w).reduceBy(reducer()))

In [ ]:
print(len(to_rem))

In [ ]:
non_rep_doms = dga_rdd.filter(lambda x: x not in to_rem)
print(non_rep_doms.collect())

In [ ]:
uniq_doms = (non_rep_doms.map(lambda w: (w, 1)).reduceByKey(lambda x,y: x+y))
print(uniq_doms.collect())

In [ ]:
print(uniq_doms)

In [ ]:
deptColumns = ["domain_name","count"]
fin = uniq_doms.toDF(deptColumns)
fin.printSchema()
fin.show(truncate=False)

fin_df = fin.toPandas()
domainf_df = fin_df['domain_name']

final_domain_li = domain_df.values.tolist()
print(final_domain_li)